## Initial EDA

In [1]:
# import libraries

import pandas as pd
import ast
import numpy as np
from shapely import Point, Polygon, LineString, union_all

In [2]:
# File Path to processed shot table

processed_output_folder = 'C:/Users/bquev/Desktop/6311 Capstone/output/processed_data/'

In [3]:
# Load the shots table from initial statsbomb load data

all_shots_df = pd.read_csv(processed_output_folder + 'shots.csv')

print(all_shots_df.shape)

(66854, 159)


In [4]:
# Review match and team coverage for each competition season

coverage_rows = []

for (competition, season), seasons_shots_df in all_shots_df.groupby(['competition_name', 'season_name']):

    # Count matches and teams per season

    matches = seasons_shots_df['match_id'].nunique()
    teams = seasons_shots_df['team_id'].nunique()

    # Find how many matches each team appears in, then keep the highest
    # If one team appears in every match, the season follows that team only

    team_matches = seasons_shots_df.groupby('team_id')['match_id'].nunique()
    most_team_matches = team_matches.max()

    # Label how much of the season the data covers

    if matches == 1:
        coverage = 'Single match'
    elif most_team_matches == matches:
        coverage = 'One team in every match'
    else:
        coverage = 'Multiple teams'

    # Save results for each competition season

    coverage_rows.append({'competition': competition,
                          'season': season,
                          'matches': matches,
                          'teams': teams,
                          'season_coverage': coverage})

# Create a single table, sorted in descending order based on matches

season_coverage_df = pd.DataFrame(coverage_rows)
season_coverage_df = season_coverage_df.sort_values('matches', ascending=False)

print(season_coverage_df.head(15))

               competition     season  matches  teams          season_coverage
47                 La Liga  2015/2016      380     20           Multiple teams
61          Premier League  2015/2016      380     20           Multiple teams
63                 Serie A  2015/2016      380     20           Multiple teams
55                 Ligue 1  2015/2016      377     20           Multiple teams
34     Indian Super league  2021/2022      115     11           Multiple teams
33          FIFA World Cup       2022       64     32           Multiple teams
32          FIFA World Cup       2018       64     32           Multiple teams
2   African Cup of Nations       2023       52     24           Multiple teams
64               UEFA Euro       2020       51     24           Multiple teams
65               UEFA Euro       2024       51     24           Multiple teams
60          Premier League  2003/2004       38     20  One team in every match
46                 La Liga  2014/2015       38     2

## Coverage of data

Based on these results, the 2015/2016 season is the only season with complete or nearly complete coverage across the four European leagues. Ligue 1 is missing three matches, but this is unlikely to significantly affect the overall shot analysis.

In [5]:
# Filter to the four European leagues for the 2015/16 season
# Premier League (2), Ligue 1 (7), La Liga (11), Serie A (12), 2015/16 season (27)

major_4_competitions_shots_df = all_shots_df.loc[
    all_shots_df['competition_id'].isin([2, 7, 11, 12])
    & (all_shots_df['season_id'] == 27)
].copy()

# Check rows and columns

print(major_4_competitions_shots_df.shape)

# Check all matches came through

print(major_4_competitions_shots_df.groupby('competition_name')['match_id'].nunique())


(37888, 159)
competition_name
La Liga           380
Ligue 1           377
Premier League    380
Serie A           380
Name: match_id, dtype: int64


## Creating shot distance and shot angle features

In [6]:
# Location is currently stored as a string x, y "[100, 50]"
# To make x and y usable, we will strip the [], split by a comma, and make the data type a float

xy = (
    major_4_competitions_shots_df['location']
    .str.strip('[]')
    .str.split(',', expand=True)
    .astype(float)
)

major_4_competitions_shots_df['x'] = xy[0]
major_4_competitions_shots_df['y'] = xy[1]

print(major_4_competitions_shots_df[['location', 'x', 'y']].head())

           location      x     y
1343  [102.7, 19.9]  102.7  19.9
1344  [107.4, 40.7]  107.4  40.7
1345  [114.5, 38.1]  114.5  38.1
1346   [97.5, 44.8]   97.5  44.8
1347  [108.6, 52.9]  108.6  52.9


In [7]:
# Since x = 120 is the goal line, and y = 40 is the center of the net,
# we use the Pythagorean theorem to find the distance of the shot to the center of the goal

major_4_competitions_shots_df['shot_distance'] = np.sqrt(
   (120 - major_4_competitions_shots_df['x'])**2
   + (40 - major_4_competitions_shots_df['y'])**2
)

print(major_4_competitions_shots_df[['x', 'y', 'shot_distance']].head())

          x     y  shot_distance
1343  102.7  19.9      26.519804
1344  107.4  40.7      12.619429
1345  114.5  38.1       5.818935
1346   97.5  44.8      23.006303
1347  108.6  52.9      17.215400


In [8]:
# Measuring the shot angle using trigonometry
# shot_angle is how wide the goal looks from the shot, in degrees

# dx is the distance to the goal line (120 - x)
# dy is the side-to-side distance from the center of the goal (y - 40)

dx = 120 - major_4_competitions_shots_df['x']
dy = major_4_competitions_shots_df['y'] - 40

# The angle to each post is found with tangent (opposite / adjacent),
# and the shot angle is the difference between the far post and near post angles.
# The tangent subtraction identity combines both angles into one fraction
# tan(B - A) = (tan B - tan A) / (1 + tan B x tan A)

# 8 * dx is the goal width times the distance to the goal line,
# the comma separates the top and bottom of the fraction
# dx**2 + dy**2 - 16 squares the distances and subtracts half the goal width squared (4² = 16)

# np.arctan2 turns the fraction back into an angle
# np.degrees converts the angle from radians to degrees

major_4_competitions_shots_df['shot_angle'] = np.degrees(
   np.arctan2(8 * dx, dx**2 + dy**2 - 16)
)

print(major_4_competitions_shots_df[['x', 'y', 'shot_distance', 'shot_angle']].head())


          x     y  shot_distance  shot_angle
1343  102.7  19.9      26.519804   11.385260
1344  107.4  40.7      12.619429   35.132707
1345  114.5  38.1       5.818935   67.907319
1346   97.5  44.8      23.006303   19.324695
1347  108.6  52.9      17.215400   18.018901


In [9]:
# Testing whether shot distance and angle work, based on a penalty shot with a known distance and angle
# Penalties are taken from the penalty spot (108, 40), so distance should be about 12 coordinate units and angle about 37 degrees

penalty_shot = major_4_competitions_shots_df[major_4_competitions_shots_df['shot_type_name'] == 'Penalty'].iloc[0]

print(penalty_shot[['player_name', 'x', 'y', 'shot_distance', 'shot_angle']])

player_name      Luis Alberto Suárez Díaz
x                                   108.3
y                                    40.3
shot_distance                   11.703846
shot_angle                      37.728461
Name: 1543, dtype: object


In [10]:
# Checking if the data has reliable distance and angles based on all penalty shots

penalty_shots_df = major_4_competitions_shots_df[
    major_4_competitions_shots_df['shot_type_name'] == 'Penalty'
]

print(penalty_shots_df[['shot_distance', 'shot_angle']].describe())

       shot_distance  shot_angle
count     400.000000  400.000000
mean       11.954634   37.002736
std         0.124881    0.353756
min        11.501739   34.454976
25%        11.900420   36.869898
50%        12.000000   36.869898
75%        12.000000   37.156345
max        12.900000   38.348424


## Shot Location, Distance and Angle

### Shot Location (x and y)

StatsBomb records shot locations using an `[x, y]` coordinate system on a standardized 120 × 80 pitch. The attacking team always shoots toward x = 120, with the center of the goal at (120, 40).

When the data was saved to CSV, the coordinates were stored as text. We converted them back into numerical values and separated them into `x` and `y` columns for analysis.

### Shot Distance

Shot distance measures how far a shot is from the center of the goal at (120, 40). We used the Pythagorean theorem to calculate the straight-line distance based on the shot's x and y coordinates.

$$\text{distance} = \sqrt{(120-x)^2+(40-y)^2}$$

Distance is measured in StatsBomb coordinate units.

### Shot Angle

Shot angle measures how wide the goal appears from where a shot is taken.

The goalposts are located at (120, 36) and (120, 44). We used these positions and the shot coordinates to calculate the angle between the two goalposts.

First, we calculated the distance to the goal line (`dx`) and the distance from the center of the goal (`dy`).

$$dx = 120 - x$$

$$dy = y - 40$$

We then used the following formula to calculate the shot angle. The formula finds the angle from the shot to each post and takes the difference between them.

$$\text{angle} = \operatorname{atan2}(8\,dx,\ dx^2+dy^2-16)$$

The 8 represents the width of the goal and 16 is half the goal width squared (4²). We used `np.arctan2()` to handle angles greater than 90 degrees and `np.degrees()` to convert the result into degrees.

### Validation

We checked our calculations using penalty kicks since they are taken from a known position at (108, 40).

At this location, the expected distance is 12 coordinate units and the expected shot angle is 36.87 degrees.

Across the 400 penalties in our dataset, the median distance was 12.0 and the median angle was 36.87 degrees. Both matched the expected results, which helps confirm that our calculations are working correctly.

**References:**

* [StatsBomb Open Data](https://github.com/hudl/open-data)
* [NumPy arctan2](https://numpy.org/doc/stable/reference/generated/numpy.arctan2.html)
* [NumPy degrees](https://numpy.org/doc/stable/reference/generated/numpy.degrees.html)
* Anthropic. (2026). *Claude* (Opus 5.5 version) [Large language model]. https://claude.ai. Used to help develop and explain the shot angle calculations.

## Selecting Columns

In [11]:
# Keep only the columns needed for shot analysis

shot_columns = [
    # IDs to link shots, players, teams and matches

    'id', 'match_id', 'player_id', 'team_id',

    # Match details

    'match_date', 'competition_name', 'season_name', 'home_team', 'away_team',

    # When the shot happened

    'period', 'minute', 'second',

    # Who took the shot

    'player_name', 'position_name', 'team_name',

    # Shot location and the features created from it

    'x', 'y', 'shot_distance', 'shot_angle',

    # Shot situation

    'play_pattern_name', 'under_pressure',
    'shot_type_name', 'shot_body_part_name', 'shot_technique_name',
    'shot_first_time', 'shot_one_on_one', 'shot_open_goal',
    'shot_aerial_won', 'shot_deflected',

    # Outcome and StatsBomb's xG for comparison

    'shot_outcome_name', 'shot_statsbomb_xg',

    # To be dropped after goalkeeper, defender and teammate features are built
    
    'shot_freeze_frame'
]

shots_analysis_df = major_4_competitions_shots_df[shot_columns].copy()

print(shots_analysis_df.shape)

(37888, 32)


In [12]:
# Check how often a penalty shot is scored

print((penalty_shots_df['shot_outcome_name'] == 'Goal').mean())

0.75


In [13]:
# Remove penalties from the analysis dataset

shots_analysis_df = shots_analysis_df[shots_analysis_df['shot_type_name'] != 'Penalty'].copy()

# Check if there are any penalty shots remaining

print(shots_analysis_df.shape)
print((shots_analysis_df['shot_type_name'] == 'Penalty').sum())

(37488, 32)
0


In [14]:
# Check how often a non-penalty shot is scored

print((shots_analysis_df['shot_outcome_name'] == 'Goal').mean())

0.09520379854886897


## Penalty Shots

Penalty shots are taken from a fixed distance and angle with only the goalkeeper defending. Since our analysis focuses on what makes a good shot during attacking play, we decided to exclude penalties. In our dataset, 75% of penalty shots resulted in goals compared to 9.5% of non-penalty shots. Including penalties could influence the model's results since they represent a different type of scoring opportunity.

## Goalies, defenders and teammates in shot lane

The 'shot_freeze_frame' column contains the positions of nearby players when a shot was taken. Since this data was saved as text in the CSV, we will convert it back into a list of players. We will then use the Shapely library to count defenders and teammates inside the shooting lane. We will also check whether the goalkeeper is in the lane.

In [15]:
# Check for missing freeze-frame data

print(shots_analysis_df['shot_freeze_frame'].isna().sum())

0


In [16]:
# Convert the freeze-frame data from text back into Python lists

shots_analysis_df['shot_freeze_frame'] = shots_analysis_df['shot_freeze_frame'].apply(ast.literal_eval)

In [17]:
# Check the first shot's freeze frame is now a list, and how many players it has

print(type(shots_analysis_df['shot_freeze_frame'].iloc[0]))
print(len(shots_analysis_df['shot_freeze_frame'].iloc[0]))

<class 'list'>
14


In [18]:
# Check if a player is inside the shooting lane or on its edge
# Polygon creates the shooting lane using the shooter and two goalposts

def in_shot_lane(player_x, player_y, shot_x, shot_y):
    lane = Polygon([(shot_x, shot_y), (120, 36), (120, 44)])
    return lane.covers(Point(player_x, player_y))

In [19]:
# Check whether the function works with a shooter at (100, 30): A is in the lane, B is outside, C is behind the shooter

print(in_shot_lane(110, 36, 100, 30))   # A: True
print(in_shot_lane(110, 40, 100, 30))   # B: False
print(in_shot_lane(95, 31, 100, 30))    # C: False

True
False
False


In [20]:
# Check each player in a shot's freeze frame
# Use in_shot_lane to count defenders and teammates inside the shooting lane
# Make a True/False for goalie

def count_players_in_shot_lane(freeze_frame, shot_x, shot_y):

    # Track defenders, teammates and the goalkeeper in the shooting lane

    defenders_in_lane = 0
    teammates_in_lane = 0
    goalkeeper_in_lane = False

    for player in freeze_frame:

        # Loop through the shot freeze frame to find the coordinates, and skip if not in shot lane

        player_x, player_y = player['location']

        if in_shot_lane(player_x, player_y, shot_x, shot_y) == False:
            continue

        # Loop through to see what the player is (defender, teammate, goalie)

        if player['teammate'] == True:
            teammates_in_lane += 1
        elif player['position']['name'] == 'Goalkeeper':
            goalkeeper_in_lane = True
        else:
            defenders_in_lane += 1
    
    return defenders_in_lane, teammates_in_lane, goalkeeper_in_lane

In [21]:
# Apply the function to each shot and save the results in three new columns

shots_analysis_df[['defenders_in_lane', 'teammates_in_lane', 'goalkeeper_in_lane']] = shots_analysis_df.apply(
    lambda row: count_players_in_shot_lane(row['shot_freeze_frame'], row['x'], row['y']),

    # Apply to every row
    axis=1,

    # Create 3 separate columns

    result_type='expand'
)

print(shots_analysis_df[['defenders_in_lane', 'teammates_in_lane', 'goalkeeper_in_lane']].head())

      defenders_in_lane  teammates_in_lane  goalkeeper_in_lane
1343                  1                  1                True
1344                  2                  0                True
1345                  1                  0                True
1346                  0                  0                True
1347                  1                  0                True


In [22]:
# Check for any empty freeze frames

print((shots_analysis_df['shot_freeze_frame'].apply(len) == 0).sum())

0


## Calculating Percent of net left after Defenders and Goalies

In [23]:
# Estimating what percentage of the net is open for shot
# Defenders and goalies block portions of the goal
# Assume defenders are 1 unit wide and goalies are 2 units wide

def percent_of_goal_open(freeze_frame, shot_x, shot_y):

    goal_line = LineString([(120, 36), (120, 44)])
    blocked_sections = []

    for player in freeze_frame:

        player_x, player_y = player['location']

        # Exclude teammates to measure blocking
        # Exclude players behind or in line with shooter, and players behind the goal line

        if player['teammate'] == True or player_x <= shot_x or player_x > 120:
            continue

        # Assume players are 1 unit wide and goalkeepers are 2 units wide
             
        if player['position']['name'] == 'Goalkeeper':
            half_width = 1.0
        else:
            half_width = 0.5

        # Use similar triangles to estimate how much of the goal each player blocks
    
        # Calculate how much the player's blocking width grows when projected onto the goal

        scale = (120 - shot_x) / (player_x - shot_x)

        # Find the center of the blocked section on the goal line

        section_center = shot_y + (player_y - shot_y) * scale

        # Calculate how far the blocked section extends on each side
            
        section_half_width = half_width * scale

        # Create a line showing the section of the goal blocked by this player
        # Subtract for the lower edge and add for the upper edge

        blocked_sections.append(LineString([
            (120, section_center - section_half_width),
            (120, section_center + section_half_width)
        ]))

    # If no opponents can block the shot, the whole goal is open

    if len(blocked_sections) == 0:
        return 100

    # Merge overlapping sections, keep only the part on the goal, and find how much is blocked

    blocked = union_all(blocked_sections).intersection(goal_line).length

    return 100 * (1 - blocked / 8)

    

In [24]:
# Calculate the percentage of goal open for every shot

shots_analysis_df['percent_of_goal_open'] = shots_analysis_df.apply(
    lambda row: percent_of_goal_open(row['shot_freeze_frame'], row['x'], row['y']),
    axis=1
)

print(shots_analysis_df['percent_of_goal_open'].describe())

count    37488.000000
mean        44.276408
std         24.729094
min          0.000000
25%         25.500000
50%         46.637308
75%         66.876335
max        100.000000
Name: percent_of_goal_open, dtype: float64


## Players in Shooting Lane Summary

The shooting lane is the triangle between the shooter and the two goalposts at (120, 36) and (120, 44). Using the player locations in 'shot_freeze_frame', we used Shapely to check which players were inside this triangle and created three features:

defenders_in_lane = Number of opponents in the lane, not including the goalkeeper.

'teammates_in_lane' = Number of the shooter's teammates in the lane.

'goalkeeper_in_lane' = Whether the opposing goalkeeper is in the lane (True or False).

The goalkeeper is tracked separately from the other defenders. We also confirmed that no shots had a missing or empty freeze frame.

## Estimated Goal Open

We also created percent_of_goal_open to estimate how much of the goal is not blocked by opponents.

Using similar triangles, we calculated how much of the goal each opponent could block based on their position. Players closer to the shooter can block a larger section of the goal.

We assumed outfield players are 1 unit wide and goalkeepers are 2 units wide. These are estimates and not actual player measurements.

We used Shapely to combine overlapping blocked sections and calculate how much of the goal remained open. The result is a percentage from 0 to 100.

This calculation only considers player positions and does not account for height, movement or shot trajectory.

**References:**

* [Python ast.literal_eval](https://docs.python.org/3/library/ast.html#ast.literal_eval)
* [Shapely documentation](https://shapely.readthedocs.io/)
* Anthropic. (2026). *Claude* (Opus 5.5 version) [Large language model]. https://claude.ai. Used to help develop the shooting lane and percentage of goal open calculation.

## Goalkeeper Positioning

The final feature we will build is goalie positioning, which can help determine how much the goalkeeper's positioning affects the likelihood of a goal.

In [25]:
# Pick opposing goalkeeper and measure 2 distances
# How far he is from center of the net and how far he is from shooter

def goalkeeper_distance(freeze_frame, shot_x, shot_y):

    for player in freeze_frame:

        if player['teammate'] == False and player['position']['name'] == 'Goalkeeper':
            goalkeeper = Point(player['location'])

            distance_from_goal = goalkeeper.distance(Point(120, 40))
            distance_to_shooter = goalkeeper.distance(Point(shot_x, shot_y))

            return distance_from_goal, distance_to_shooter

    # If goalkeeper is not in the freeze frame, return None

    return None, None



In [26]:
# Calculate both goalkeeper distances for every shot

shots_analysis_df[['goalkeeper_distance_from_goal', 'goalkeeper_distance_to_shooter']] = shots_analysis_df.apply(
    lambda row: goalkeeper_distance(row['shot_freeze_frame'], row['x'], row['y']),
    axis=1,
    result_type='expand'
)

print(shots_analysis_df[['goalkeeper_distance_from_goal', 'goalkeeper_distance_to_shooter']].describe())

       goalkeeper_distance_from_goal  goalkeeper_distance_to_shooter
count                   37456.000000                    37456.000000
mean                        3.497600                       16.423247
std                         2.411816                        8.829839
min                         0.100000                        0.360555
25%                         2.024846                        9.097252
50%                         3.047950                       15.479664
75%                         4.215448                       23.181242
max                        70.377837                       74.030872


In [27]:
# Shots where the goalkeeper is missing from the freeze frame

missing_goalkeeper = shots_analysis_df['goalkeeper_distance_from_goal'].isna()

print(missing_goalkeeper.sum())

32


In [28]:
# Check shot outcomes based on missing goalkeeper data

print(shots_analysis_df.loc[missing_goalkeeper, ['shot_type_name', 'shot_open_goal', 'shot_outcome_name']].value_counts(dropna=False))

shot_type_name  shot_open_goal  shot_outcome_name
Open Play       NaN             Saved                19
                                Goal                 11
                                Blocked               2
Name: count, dtype: int64


In [29]:
# Drop shots where the goalkeeper is missing from the freeze frame
# These are missing data, not open goals (19 of the 32 were saved, which requires a goalkeeper)

shots_analysis_df = shots_analysis_df[shots_analysis_df['goalkeeper_distance_from_goal'].notna()].copy()

print(shots_analysis_df.shape)

(37456, 38)


## Summary of new features

| Feature | What it measures | Type |
|---|---|---|
| `shot_distance` | Distance to the center of the goal | Number |
| `shot_angle` | How wide the goal looks from the shot | Degrees |
| `defenders_in_lane` | Outfield opponents in the shooting lane | Count |
| `teammates_in_lane` | Teammates in the shooting lane | Count |
| `goalkeeper_in_lane` | Whether the goalkeeper is in the lane | True/False |
| `percent_of_goal_open` | Share of the goal not blocked by opponents | 0–100 |
| `goalkeeper_distance_from_goal` | Distance between the goalkeeper and the center of the goal | Number |
| `goalkeeper_distance_to_shooter` | How close the goalkeeper is to the shooter | Number |

### Feature Engineering Summary

We created several new features to see how shot distance and angle affect the probability of scoring. We also built shooting lane features to see whether having players between the shooter and the goal makes a difference.

While working on shot angles, we decided to use geometry to estimate how much of the goal is blocked by defenders and the goalkeeper. We used this to calculate the percentage of the goal that remains open for a shot. We also created goalkeeper-specific distance features to see whether the goalkeeper's positioning affects the chance of scoring.

### Limitations

One limitation is that our calculations assume shots travel in a straight line. This means curling shots around defenders, shots over defensive walls, and chipped shots are treated the same as straight shots.

We also assumed outfield players are 1 unit wide and goalkeepers are 2 units wide. These are estimates rather than actual player measurements.

Football pitches are not all the same size. Because of this, we used StatsBomb's standardized 120 × 80 coordinate system. Our distances are measured in coordinate units rather than yards or meters to keep the calculations consistent across matches.

Finally, 32 shots were removed because the opposing goalkeeper was missing from the freeze frame, even though 19 of those shots were recorded as saves. Since these records contained clear inconsistencies and made up such a small portion of the dataset, we decided to exclude them from the analysis.

In [30]:
# Any new freeze frame features should be added above this cell
# Drop shot_freeze_frame now that all freeze frame features have been created

shots_analysis_df = shots_analysis_df.drop(columns=['shot_freeze_frame'])

print(shots_analysis_df.shape)

(37456, 37)


## Data Quality Checks

In [31]:
# Check for NA's

shots_analysis_df.isna().sum().sort_values(ascending=False)

shot_open_goal                    37068
shot_deflected                    36940
shot_one_on_one                   35599
shot_aerial_won                   34100
under_pressure                    27652
shot_first_time                   25604
match_date                            0
player_id                             0
match_id                              0
id                                    0
team_id                               0
home_team                             0
competition_name                      0
player_name                           0
position_name                         0
x                                     0
season_name                           0
period                                0
minute                                0
second                                0
away_team                             0
play_pattern_name                     0
shot_angle                            0
shot_distance                         0
y                                     0


In [32]:
# StatsBomb data only records True flags
# Have to make blank values false for the missing values

missing_false_flag = [
    'under_pressure', 'shot_first_time', 'shot_one_on_one',
    'shot_open_goal', 'shot_aerial_won', 'shot_deflected'
]

shots_analysis_df[missing_false_flag] = shots_analysis_df[missing_false_flag].fillna(False).astype(bool)

# Check if NA's were all fixed

shots_analysis_df.isna().sum().sort_values(ascending=False)

id                                0
match_id                          0
player_id                         0
team_id                           0
match_date                        0
competition_name                  0
season_name                       0
home_team                         0
away_team                         0
period                            0
minute                            0
second                            0
player_name                       0
position_name                     0
team_name                         0
x                                 0
y                                 0
shot_distance                     0
shot_angle                        0
play_pattern_name                 0
under_pressure                    0
shot_type_name                    0
shot_body_part_name               0
shot_technique_name               0
shot_first_time                   0
shot_one_on_one                   0
shot_open_goal                    0
shot_aerial_won             

In [33]:
# Check for any values that are not possible
# Such as location off pitch, negative distances, more than 22 players etc

shots_analysis_df[[
    'x', 'y', 'shot_distance', 'shot_angle', 'percent_of_goal_open',
    'defenders_in_lane', 'teammates_in_lane',
    'goalkeeper_distance_from_goal', 'goalkeeper_distance_to_shooter'
]].describe()

,x,y,shot_distance,shot_angle,percent_of_goal_open,defenders_in_lane,teammates_in_lane,goalkeeper_distance_from_goal,goalkeeper_distance_to_shooter
count,37456.000000,37456.000000,37456.000000,37456.000000,37456.000000,37456.000000,37456.000000,37456.000000,37456.000000
mean,103.159379,39.741190,19.639594,24.740879,44.253006,0.771652,0.188728,3.497600,16.423247
std,8.830798,10.022854,8.740579,15.347882,24.712280,0.969704,0.477749,2.411816,8.829839
min,35.400000,0.300000,0.632456,-0.060911,0.000000,0.000000,0.000000,0.100000,0.360555
25%,96.700000,32.300000,12.484991,14.826171,25.496201,0.000000,0.000000,2.024846,9.097252
50%,104.300000,39.700000,19.231485,19.122785,46.615896,1.000000,0.000000,3.047950,15.479664
75%,110.300000,47.200000,26.043233,30.053708,66.860465,1.000000,0.000000,4.215448,23.181242
max,120.200000,79.600000,88.833158,180.000000,100.000000,8.000000,5.000000,70.377837,74.030872


In [34]:
# Shots taken from behind the goal line (x over 120) or with a negative angle

shots_analysis_df[(shots_analysis_df['x'] > 120) | (shots_analysis_df['shot_angle'] < 0)][
    ['x', 'y', 'shot_angle', 'shot_type_name', 'shot_outcome_name']
]

,x,y,shot_angle,shot_type_name,shot_outcome_name
11391,120.2,1.0,-0.060911,Corner,Goal


In [35]:
# Count shots taken directly from a corner and how many were scored

corner_shots = shots_analysis_df[shots_analysis_df['shot_type_name'] == 'Corner']

print('Shots directly from a corner:', len(corner_shots))
print('Goals directly from a corner:', (corner_shots['shot_outcome_name'] == 'Goal').sum())

Shots directly from a corner: 7
Goals directly from a corner: 3


In [36]:
# Exclude shots directly taken from corners since there are only 7 shots

shots_analysis_df = shots_analysis_df[shots_analysis_df['shot_type_name'] != 'Corner'].copy()

shots_analysis_df.shape

(37449, 37)

In [37]:
# Shots with a very wide angle (over 150 degrees), taken almost on the goal line

shots_analysis_df[shots_analysis_df['shot_angle'] > 150][
    ['x', 'y', 'shot_distance', 'shot_angle', 'shot_body_part_name', 'shot_outcome_name']
]

,x,y,shot_distance,shot_angle,shot_body_part_name,shot_outcome_name
1435,119.3,38.1,2.024846,154.798876,Right Foot,Goal
2083,120.0,43.7,3.700000,180.000000,Right Foot,Goal
14417,119.3,40.6,0.921954,159.713824,Right Foot,Goal
22646,120.0,42.0,2.000000,180.000000,Head,Goal
24858,119.4,40.2,0.632456,162.897271,Right Foot,Goal
27067,119.7,42.0,2.022375,168.606829,Right Foot,Goal
29956,119.2,38.0,2.154066,150.603947,Right Foot,Goal
40234,119.0,39.0,1.414214,150.255119,Head,Goal
43728,120.0,37.8,2.200000,180.000000,Right Foot,Goal
44766,119.4,38.7,1.431782,161.012376,Right Foot,Goal


In [38]:
# Summary of shots taken from the shooter's own half

own_half_shots = shots_analysis_df[shots_analysis_df['x'] < 60]

print('Shots from own half:', len(own_half_shots))
print(own_half_shots['shot_outcome_name'].value_counts())
print(own_half_shots['goalkeeper_distance_from_goal'].describe())

Shots from own half: 44
shot_outcome_name
Wayward             19
Off T               14
Saved                7
Saved Off Target     3
Blocked              1
Name: count, dtype: int64
count    44.000000
mean     14.522014
std      12.524821
min       2.729469
25%       7.966659
50%      12.106718
75%      16.106424
max      70.377837
Name: goalkeeper_distance_from_goal, dtype: float64


In [39]:
# Shots with 5 or more defenders in the lane, by shot type

print(shots_analysis_df[shots_analysis_df['defenders_in_lane'] >= 5]['shot_type_name'].value_counts())

shot_type_name
Free Kick    104
Open Play     95
Name: count, dtype: int64


In [40]:
# Distance of open play shots with 5 or more defenders in the lane

crowded_open_play = shots_analysis_df[
    (shots_analysis_df['defenders_in_lane'] >= 5) & (shots_analysis_df['shot_type_name'] == 'Open Play')
]

In [41]:
# Check the median shot distance and goal rate for crowded open-play shots

print('Median distance:', round(crowded_open_play['shot_distance'].median(), 1))
print(f"Goal rate: {(crowded_open_play['shot_outcome_name'] == 'Goal').mean():.1%}")

Median distance: 21.5
Goal rate: 3.2%


In [42]:
# Shots with 3 or more teammates in the lane, by play pattern

print(shots_analysis_df[shots_analysis_df['teammates_in_lane'] >= 3]['play_pattern_name'].value_counts())

play_pattern_name
From Free Kick    89
From Corner       55
From Throw In      4
Regular Play       3
From Goal Kick     1
Name: count, dtype: int64


In [43]:
# Check that every shot appears only once

print(shots_analysis_df['id'].duplicated().sum())

0


In [44]:
# Create the target column: True if the shot was a goal

shots_analysis_df['is_goal'] = shots_analysis_df['shot_outcome_name'] == 'Goal'

print(shots_analysis_df['is_goal'].mean())

0.09492910358086998


## Data Quality Summary

We reviewed the dataset for missing values, duplicate shots, and possible errors in our new features. Some Boolean columns had missing values because StatsBomb only records them when they are True. We filled these with False and confirmed there were no remaining missing values or duplicate shot IDs.

We also investigated shots with extreme distances, angles, and goalkeeper positions, along with shots that had several players in the shooting lane. All of these observations made sense when we looked at the situations. For example, goalkeepers positioned far from their goal helped explain some shots taken from a team's own half. We also found shots with angles reaching 180°, which occurred when players were on the goal line between the posts. Since these were realistic situations, we decided to keep them.

We did remove seven shots taken directly from corners because they represented a different type of scoring opportunity. One was also recorded beyond the goal line, resulting in a negative shot angle.

After these checks, our dataset contained 37,449 shots. We created `is_goal` as our target variable, with 9.49% of shots resulting in goals. We also kept `shot_statsbomb_xg` and `shot_deflected` for comparison and exploratory analysis, but they will not be used as model inputs.